# Wan VACE-1.3B Image-to-Video — free Colab T4 (16 GB)

Turns one product photo into a **9:16 vertical MP4** (480x832, 16 fps) with the **1.3B** Wan model, using the **official [`Wan-Video/Wan2.1`](https://github.com/Wan-Video/Wan2.1) repository** and its own `generate.py` (task `vace-1.3B`), pinned to a specific upstream commit. No Diffusers pipeline.

**To run it:** `Runtime -> Run all`. There is one code cell. It checks the GPU, asks for **one** product photo (the only click), installs a few small packages, clones the repo, downloads the model (~19 GB, first run only), generates the clip, plays it and downloads `output.mp4`. Nothing needs editing and no runtime restart is needed.

## Which model, and why not "Wan 2.2 I2V-1.3B"

That model does not exist: the Wan 2.2 repository only ships 14B and 5B models, and Wan-AI publishes no I2V-1.3B. **Wan2.1 VACE-1.3B** is the only 1.3B model that accepts an input image. The notebook uses it in its first-frame mode: your photo becomes frame 0 (fitted to 9:16 with a blurred backdrop so the whole product stays visible) and the model generates the rest of the clip from it.

## What this notebook changes around the official code

`generate.py` and the model code are used unmodified. A small launcher applies four in-memory patches before running it, because the stock code cannot run on a free T4:

| Problem in the stock code | Fix |
|---|---|
| The T5 text encoder is built on the CPU and its 11 GB checkpoint is loaded on top of it (~23 GB RAM, free Colab has ~12.7 GB). | The official T5 code encodes the prompt **once, in its own process**, with the weights memory-mapped straight onto the GPU. The video model then reads those saved embeddings. The T5 never coexists with the video model. |
| Attention requires `flash_attn`, which does not run on a T4. | PyTorch `scaled_dot_product_attention` (memory-efficient kernel), with the text padding mask kept. |
| The model runs in bf16, which a T4 only emulates. | fp16 (weights cast once, ~3.6 GB on the GPU). NaNs are checked every step; on overflow it retries in bf16 at a shorter length. |
| `decord` has no Python 3.13 wheel. | The first-frame video and mask are handed over from memory through a tiny stand-in for `decord`; nothing is decoded from a file. |

Also: `flash_attn`, `dashscope` and the rest of `requirements.txt` are not installed (torch, numpy and transformers are never touched), and `--offload_model True` is always on.

## Automatic fallback

Each attempt runs in a fresh process. If the GPU runs out of memory it steps down through 49, 33, then 17 frames. Frame counts are always 4n+1. Prompt, frames, steps, guidance and seed are form fields at the top of the cell; the defaults work as they are (49 frames is about 3 s).

Output is 480x832 because that is the only portrait size this model supports.

In [ ]:
#@title Wan VACE-1.3B image-to-video (official Wan-Video/Wan2.1 repo) on a free T4  (Runtime > Run all)
# One cell does everything: checks the GPU, asks for ONE product photo, installs, clones the
# official repo, downloads the model, generates a 9:16 MP4 and downloads it. No editing needed.
import os
import shutil
import subprocess
import sys
import threading
import time

PROMPT = "Cinematic product advertisement: the product stays sharp and centered, slow smooth camera push-in, soft studio lighting, subtle natural motion, high quality commercial video"  #@param {type:"string"}
FRAMES = 49  #@param {type:"integer"}
NUM_STEPS = 30  #@param {type:"integer"}
GUIDANCE_SCALE = 5.0  #@param {type:"number"}
SEED = 42  #@param {type:"integer"}

REPO_URL = "https://github.com/Wan-Video/Wan2.1"
REPO_COMMIT = "9737cba9c1c3c4d04b33fcad41c111989865d315"
MODEL_REPO = "Wan-AI/Wan2.1-VACE-1.3B"
ALLOW_PATTERNS = ["*.safetensors", "*.pth", "config.json", "google/*"]
REQUIRED_FILES = {
    "diffusion_pytorch_model.safetensors": 7.0e9,
    "models_t5_umt5-xxl-enc-bf16.pth": 11.3e9,
    "Wan2.1_VAE.pth": 5.0e8,
    "config.json": 100,
    "google/umt5-xxl/spiece.model": 1e6,
    "google/umt5-xxl/tokenizer.json": 1e6,
    "google/umt5-xxl/tokenizer_config.json": 1000,
}
FPS = 16                    # the official sample_fps for this model
SIZE = "480*832"            # width*height: the model's 9:16 portrait size
BF16_MAX_FRAMES = 33        # bf16 attention runs one head at a time on a T4, so keep it short

WORK = "/content" if os.path.isdir("/content") else os.getcwd()
REPO_DIR = os.path.join(WORK, "Wan2.1")
CKPT_DIR = os.path.join(WORK, "Wan2.1-VACE-1.3B")
LAUNCHER_PATH = os.path.join(WORK, "wan_t4_launcher.py")
EMBEDS_PATH = os.path.join(WORK, "prompt_embeds.pt")
OUTPUT = os.path.join(WORK, "output.mp4")

EXIT_OK, EXIT_OOM, EXIT_NAN, EXIT_ENCODER = 0, 3, 4, 7

LAUNCHER_SOURCE = r'''
import argparse
import gc
import os
import runpy
import sys
import time
import traceback
import types

import numpy as np
import torch

EXIT_OOM, EXIT_NAN, EXIT_ENCODER = 3, 4, 7
FPS = 16
SIZE_W, SIZE_H = 480, 832


class NanOutput(RuntimeError):
    pass


def log(msg):
    print(time.strftime("[%H:%M:%S] ") + str(msg), flush=True)


def is_oom(exc):
    return isinstance(exc, torch.cuda.OutOfMemoryError) or "out of memory" in str(exc).lower()


def first_frame(path, width, height):
    """Product photo -> exactly width x height. Keeps the whole product (blurred backdrop) unless the ratio is already ~9:16."""
    from PIL import Image, ImageFilter, ImageOps
    img = ImageOps.exif_transpose(Image.open(path)).convert("RGB")
    iw, ih = img.size
    target = width / height
    if abs(iw / ih - target) / target < 0.08:
        return ImageOps.fit(img, (width, height), method=Image.LANCZOS)
    canvas = ImageOps.fit(img, (width, height), method=Image.LANCZOS).filter(ImageFilter.GaussianBlur(28))
    scale = min(width / iw, height / ih)
    fg = img.resize((max(1, round(iw * scale)), max(1, round(ih * scale))), Image.LANCZOS)
    canvas.paste(fg, ((width - fg.width) // 2, (height - fg.height) // 2))
    return canvas


def install_import_shims(sources):
    """Stand-ins so the official code imports on Python 3.13 / Colab. `sources` maps mem:// keys to uint8 (F,H,W,3) arrays."""
    class MemReader:
        def __init__(self, key):
            self.frames = sources[key]

        def __len__(self):
            return len(self.frames)

        def get_avg_fps(self):
            return float(FPS)

        def get_frame_timestamp(self, i):
            return np.array([i / FPS, (i + 1) / FPS], dtype=np.float32)

        def next(self):
            return torch.from_numpy(self.frames[0])

        def get_batch(self, ids):
            return torch.from_numpy(self.frames[np.asarray(ids)])

    decord = types.ModuleType("decord")
    decord.VideoReader = MemReader
    decord.bridge = types.SimpleNamespace(set_bridge=lambda name: None)
    sys.modules["decord"] = decord
    try:
        __import__("dashscope")
    except Exception:
        sys.modules["dashscope"] = types.ModuleType("dashscope")


def make_attention(work_dtype):
    import torch.nn.functional as F

    def attention(q, k, v, q_lens=None, k_lens=None, dropout_p=0.0, softmax_scale=None, q_scale=None,
                  causal=False, window_size=(-1, -1), deterministic=False, dtype=None, version=None):
        out_dtype = q.dtype
        if q_scale is not None:
            q = q * q_scale
        q = q.transpose(1, 2).to(work_dtype)
        k = k.transpose(1, 2).to(work_dtype)
        v = v.transpose(1, 2).to(work_dtype)
        mask = None
        if k_lens is not None:
            length = k.size(2)
            lens = k_lens.to(k.device)
            if bool((lens < length).any()):
                mask = (torch.arange(length, device=k.device)[None, :] < lens[:, None])[:, None, None, :]
        if work_dtype == torch.float16:
            out = F.scaled_dot_product_attention(q, k, v, attn_mask=mask, is_causal=causal, scale=softmax_scale)
        else:
            heads = [
                F.scaled_dot_product_attention(q[:, h:h + 1], k[:, h:h + 1], v[:, h:h + 1],
                                               attn_mask=mask, is_causal=causal, scale=softmax_scale)
                for h in range(q.size(1))
            ]
            out = torch.cat(heads, dim=1)
        return out.transpose(1, 2).contiguous().to(out_dtype)

    return attention


def apply_t4_patches(work_dtype, embeds_path):
    import wan
    import wan.modules
    import wan.modules.attention as attention_module
    import wan.modules.model as model_module
    import wan.modules.vace_model as vace_module
    from wan.configs import WAN_CONFIGS
    from wan.modules.t5 import T5EncoderModel

    # 1. flash_attn cannot run on a T4: use PyTorch SDPA (memory-efficient kernel in fp16).
    attention = make_attention(work_dtype)
    attention_module.flash_attention = attention
    attention_module.attention = attention
    model_module.flash_attention = attention
    wan.modules.flash_attention = attention

    # 2. T4 has no fast bf16: run the DiT in fp16 (weights cast once, so the GPU holds ~3.6 GB, not 7 GB).
    WAN_CONFIGS["vace-1.3B"].param_dtype = work_dtype
    original_from_pretrained = vace_module.VaceWanModel.from_pretrained.__func__

    def from_pretrained(cls, *args, **kwargs):
        kwargs.setdefault("torch_dtype", work_dtype)
        return original_from_pretrained(cls, *args, **kwargs).to(work_dtype)

    vace_module.VaceWanModel.from_pretrained = classmethod(from_pretrained)

    # 3. Fail fast on NaN/inf instead of finishing a black video.
    original_forward = vace_module.VaceWanModel.forward
    state = {"calls": 0}

    def checked_forward(self, *args, **kwargs):
        out = original_forward(self, *args, **kwargs)
        if not bool(torch.isfinite(out[0]).all()):
            raise NanOutput("DiT output became NaN/inf at call %d" % (state["calls"] + 1))
        state["calls"] += 1
        if state["calls"] == 1:
            log("first DiT pass ok, peak VRAM %.1f GiB" % (torch.cuda.max_memory_allocated() / 2 ** 30))
        return out

    vace_module.VaceWanModel.forward = checked_forward

    # 4. The official T5EncoderModel builds 11 GB on the CPU and loads a second 11 GB copy (~23 GB RAM),
    #    which free Colab cannot hold. Prompts were already encoded by the official T5 code in a separate
    #    process (stage "encode"); serve those embeddings here.
    class NoWeights:
        def to(self, *args, **kwargs):
            return self

        def cpu(self):
            return self

    def cached_init(self, text_len, dtype=None, device=None, checkpoint_path=None, tokenizer_path=None, shard_fn=None):
        self.text_len = text_len
        self.dtype = dtype
        self.device = device
        self.cache = torch.load(embeds_path, map_location="cpu")
        self.model = NoWeights()

    def cached_call(self, texts, device):
        out = []
        for text in texts:
            if text not in self.cache:
                raise KeyError("prompt was not pre-encoded: %r" % text[:60])
            out.append(self.cache[text].to(device=device, dtype=torch.float32))
        return out

    T5EncoderModel.__init__ = cached_init
    T5EncoderModel.__call__ = cached_call


def stage_check(a):
    sys.path.insert(0, a.repo)
    os.chdir(a.repo)
    install_import_shims({})
    import importlib
    for name in ("wan", "wan.configs", "wan.utils.prompt_extend", "wan.utils.utils"):   # what generate.py imports
        importlib.import_module(name)
    assert "480*832" in importlib.import_module("wan.configs").SUPPORTED_SIZES["vace-1.3B"]
    log("official Wan2.1 modules import fine (vace-1.3B supports 480*832)")
    return 0


def stage_encode(a):
    sys.path.insert(0, a.repo)
    os.chdir(a.repo)
    install_import_shims({})
    from wan.configs import WAN_CONFIGS
    from wan.modules.t5 import T5EncoderModel, umt5_xxl
    from wan.modules.tokenizers import HuggingfaceTokenizer

    cfg = WAN_CONFIGS["vace-1.3B"]
    texts = [a.prompt, cfg.sample_neg_prompt]
    ckpt = os.path.join(a.ckpt, cfg.t5_checkpoint)
    tokenizer_path = os.path.join(a.ckpt, cfg.t5_tokenizer)
    log("T5: building the official umt5-xxl encoder without allocating RAM, then memory-mapping the weights")
    model = umt5_xxl(encoder_only=True, return_tokenizer=False, dtype=cfg.t5_dtype, device="meta").eval().requires_grad_(False)
    try:
        state_dict = torch.load(ckpt, map_location="cpu", mmap=True, weights_only=True)
    except Exception as exc:
        log("mmap load unavailable (%s); loading normally" % type(exc).__name__)
        state_dict = torch.load(ckpt, map_location="cpu", weights_only=True)
    model.load_state_dict(state_dict, assign=True)
    del state_dict
    model.to("cuda")
    log("T5: %.1f GiB on the GPU, encoding the prompts" % (torch.cuda.memory_allocated() / 2 ** 30))

    encoder = T5EncoderModel.__new__(T5EncoderModel)   # official __call__, official weights, no official __init__
    encoder.text_len = cfg.text_len
    encoder.dtype = cfg.t5_dtype
    encoder.device = torch.device("cuda")
    encoder.checkpoint_path = ckpt
    encoder.tokenizer_path = tokenizer_path
    encoder.model = model
    encoder.tokenizer = HuggingfaceTokenizer(name=tokenizer_path, seq_len=cfg.text_len, clean="whitespace")
    with torch.no_grad():
        contexts = encoder(texts, torch.device("cuda"))
    cache = {}
    for text, ctx in zip(texts, contexts):
        if not bool(torch.isfinite(ctx.float()).all()):
            raise RuntimeError("T5 produced non-finite embeddings")
        cache[text] = ctx.detach().cpu()
    torch.save(cache, a.embeds)
    del encoder, model, contexts
    gc.collect()
    torch.cuda.empty_cache()
    log("T5: %d prompt embeddings saved, encoder released" % len(cache))
    return 0


def stage_generate(a):
    work_dtype = torch.float16 if a.dtype == "float16" else torch.bfloat16
    sys.path.insert(0, a.repo)
    os.chdir(a.repo)
    frames = a.frames
    frame0 = np.asarray(first_frame(a.image, SIZE_W, SIZE_H), dtype=np.uint8)
    video = np.full((frames, SIZE_H, SIZE_W, 3), 127, dtype=np.uint8)
    video[0] = frame0
    mask = np.full((frames, SIZE_H, SIZE_W, 3), 255, dtype=np.uint8)
    mask[0] = 0                       # black = keep (the photo), white = generate
    install_import_shims({"mem://video": video, "mem://mask": mask})
    apply_t4_patches(work_dtype, a.embeds)
    log("official generate.py: task vace-1.3B, %s, %d frames, %d steps, %s" % ("480*832", frames, a.steps, a.dtype))
    sys.argv = [
        "generate.py", "--task", "vace-1.3B", "--size", "480*832", "--ckpt_dir", a.ckpt,
        "--src_video", "mem://video", "--src_mask", "mem://mask",
        "--prompt", a.prompt, "--frame_num", str(frames), "--sample_steps", str(a.steps),
        "--sample_guide_scale", str(a.guidance), "--base_seed", str(a.seed),
        "--offload_model", "True", "--save_file", a.out,
    ]
    runpy.run_path("generate.py", run_name="__main__")
    if not os.path.exists(a.out) or os.path.getsize(a.out) < 10000:
        raise RuntimeError("generate.py finished but no MP4 was written")
    log("saved %s (%.1f MB)" % (a.out, os.path.getsize(a.out) / 1e6))
    return 0


def main():
    p = argparse.ArgumentParser()
    p.add_argument("--stage", choices=["check", "encode", "generate"], required=True)
    p.add_argument("--repo", required=True)
    p.add_argument("--ckpt", default="")
    p.add_argument("--image", default="")
    p.add_argument("--out", default="")
    p.add_argument("--embeds", default="")
    p.add_argument("--prompt", default="")
    p.add_argument("--frames", type=int, default=49)
    p.add_argument("--steps", type=int, default=30)
    p.add_argument("--guidance", type=float, default=5.0)
    p.add_argument("--seed", type=int, default=42)
    p.add_argument("--dtype", choices=["float16", "bfloat16"], default="float16")
    a = p.parse_args()
    stages = {"check": stage_check, "encode": stage_encode, "generate": stage_generate}
    try:
        return stages[a.stage](a)
    except NanOutput as exc:
        log("NAN: %s" % exc)
        return EXIT_NAN
    except Exception as exc:
        if is_oom(exc):
            log("OOM: %s" % str(exc).splitlines()[0])
            return EXIT_OOM
        traceback.print_exc()
        return EXIT_ENCODER if a.stage == "encode" else 1


if __name__ == "__main__":
    sys.exit(main())
'''

CHECK_CODE = r'''
import torch, torchvision, transformers, diffusers, easydict, einops, ftfy, regex, imageio, imageio_ffmpeg
from diffusers.configuration_utils import ConfigMixin, register_to_config
from diffusers.models.modeling_utils import ModelMixin
from diffusers.schedulers.scheduling_utils import KarrasDiffusionSchedulers, SchedulerMixin, SchedulerOutput
from diffusers.utils import deprecate, is_scipy_available
from diffusers.utils.torch_utils import randn_tensor
from PIL import Image
print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| diffusers", diffusers.__version__, "| transformers", transformers.__version__)
assert torch.cuda.is_available(), "torch cannot see the GPU"
'''


def stamp(msg):
    print(time.strftime("[%H:%M:%S] ") + msg, flush=True)


def run_streamed(cmd, env=None, cwd=None):
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1, env=env, cwd=cwd)
    for line in proc.stdout:
        print(line, end="", flush=True)
    return proc.wait()


def run_captured(cmd, cwd=None):
    proc = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, cwd=cwd)
    return proc.returncode, proc.stdout.strip()


def build_ladder(frames):
    """Frame counts to try, longest first. Every entry is 4n+1, as the model requires."""
    frames = max(5, ((int(frames) - 1) // 4) * 4 + 1)
    return [frames] + [n for n in (81, 65, 49, 33, 17) if n < frames]


def run_ladder(ladder, run_rung):
    """run_rung(frames, dtype) -> exit code. Returns (frames, dtype) on success, else None."""
    dtype = "float16"
    i = 0
    while i < len(ladder):
        frames = ladder[i]
        if dtype == "bfloat16" and frames > BF16_MAX_FRAMES:
            i += 1
            continue
        code = run_rung(frames, dtype)
        if code == EXIT_OK:
            return frames, dtype
        if code == EXIT_OOM:
            stamp("out of GPU memory at %d frames -> trying a shorter clip" % frames)
            i += 1
        elif code == EXIT_NAN and dtype == "float16":
            stamp("fp16 overflowed (NaN) -> retrying in bf16 with a shorter clip")
            dtype = "bfloat16"
        else:
            stamp("generation failed with exit code %s (details above)" % code)
            return None
    return None


def verify_checkpoint(ckpt_dir):
    """List of problems (empty when every file the model needs is present and full-sized)."""
    problems = []
    for name, min_bytes in REQUIRED_FILES.items():
        path = os.path.join(ckpt_dir, name)
        if not os.path.exists(path):
            problems.append("missing " + name)
        elif os.path.getsize(path) < min_bytes:
            problems.append("%s is only %.2f GB" % (name, os.path.getsize(path) / 1e9))
    return problems


def dir_size(path):
    total = 0
    for root, _dirs, names in os.walk(path):
        for name in names:
            try:
                total += os.path.getsize(os.path.join(root, name))
            except OSError:
                pass
    return total


def prepare_repo():
    if os.path.exists(os.path.join(REPO_DIR, "generate.py")) and os.path.exists(os.path.join(REPO_DIR, "wan", "vace.py")):
        print("official repo already present:", REPO_DIR)
        return
    shutil.rmtree(REPO_DIR, ignore_errors=True)
    os.makedirs(REPO_DIR)
    steps = [
        ["git", "init", "-q"],
        ["git", "remote", "add", "origin", REPO_URL + ".git"],
        ["git", "fetch", "-q", "--depth", "1", "origin", REPO_COMMIT],
        ["git", "checkout", "-q", "--detach", "FETCH_HEAD"],
    ]
    if any(run_streamed(cmd, cwd=REPO_DIR) != 0 for cmd in steps):
        print("pinned commit not fetchable; cloning the default branch instead")
        shutil.rmtree(REPO_DIR, ignore_errors=True)
        if run_streamed(["git", "clone", "-q", "--depth", "1", REPO_URL + ".git", REPO_DIR]) != 0:
            raise SystemExit("Could not clone " + REPO_URL)
    if not os.path.exists(os.path.join(REPO_DIR, "wan", "vace.py")):
        raise SystemExit("The cloned repository has no wan/vace.py - it is not the expected Wan2.1 code.")
    _code, head = run_captured(["git", "rev-parse", "HEAD"], cwd=REPO_DIR)
    print("official repo:", REPO_URL, "@", head)


def main():
    started = time.time()
    stamp("Step 1/7  checking the runtime")
    code, smi = run_captured(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv,noheader,nounits"])
    if code != 0 or not smi:
        raise SystemExit("No GPU found. Use Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")
    name, total_mib, used_mib = [part.strip() for part in smi.splitlines()[0].split(",")]
    print("GPU: %s | %.1f GiB VRAM (%s MiB already in use)" % (name, float(total_mib) / 1024, used_mib))
    have_model = os.path.isdir(CKPT_DIR) and not verify_checkpoint(CKPT_DIR)
    if not have_model and shutil.disk_usage(WORK).free < 25 * 10 ** 9:
        raise SystemExit("Not enough free disk (need ~25 GB). Runtime > Disconnect and delete runtime, then Run all.")

    stamp("Step 2/7  upload ONE product photo (the only thing you need to click)")
    image_path = None
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            original = sorted(uploaded)[0]
            ext = os.path.splitext(original)[1].lower() or ".jpg"
            image_path = os.path.join(WORK, "input_image" + ext)
            with open(image_path, "wb") as fh:
                fh.write(uploaded[original])
    except ImportError:
        pass
    if image_path is None:
        older = [os.path.join(WORK, f) for f in os.listdir(WORK) if f.startswith("input_image.")]
        if not older:
            raise SystemExit("No image was uploaded. Run the cell again and choose a product photo.")
        image_path = older[0]
        print("no new upload - reusing", image_path)
    from PIL import Image
    with Image.open(image_path) as probe:
        probe.verify()
    print("image ok:", image_path)

    stamp("Step 3/7  installing the small set of missing packages (child processes - no restart needed)")
    pip = [sys.executable, "-m", "pip", "install", "-q", "--disable-pip-version-check"]
    plans = [
        ["easydict", "einops", "ftfy", "regex", "imageio", "imageio-ffmpeg", "diffusers"],
        ["-U", "diffusers", "transformers"],
    ]
    ready = False
    for extra in plans:
        if run_streamed(pip + extra) != 0:
            print("pip step failed; trying the next option")
            continue
        code, out = run_captured([sys.executable, "-c", CHECK_CODE])
        print(out.splitlines()[-1] if out else "")
        if code == 0:
            ready = True
            break
    if not ready:
        raise SystemExit("Dependencies could not be made importable. The last message is printed above.")

    stamp("Step 4/7  cloning the official Wan-Video/Wan2.1 repository and checking it imports")
    prepare_repo()
    with open(LAUNCHER_PATH, "w", encoding="utf-8") as fh:
        fh.write(LAUNCHER_SOURCE)
    env = dict(os.environ, PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True", PYTHONUNBUFFERED="1", TOKENIZERS_PARALLELISM="false")
    if run_streamed([sys.executable, LAUNCHER_PATH, "--stage", "check", "--repo", REPO_DIR], env=env) != 0:
        raise SystemExit("The official repo does not import in this environment (see the traceback above).")

    stamp("Step 5/7  downloading the model (~19 GB, resumable)")
    for attempt in (1, 2):
        problems = verify_checkpoint(CKPT_DIR) if os.path.isdir(CKPT_DIR) else ["not downloaded"]
        if not problems:
            print("model files are all present")
            break
        if attempt == 2:
            raise SystemExit("Model download is incomplete: " + "; ".join(problems))
        stop = threading.Event()

        def monitor():
            while not stop.wait(20):
                print("   downloaded %.1f GB of ~19 GB" % (dir_size(CKPT_DIR) / 1e9), flush=True)

        threading.Thread(target=monitor, daemon=True).start()
        download_code = (
            "import os, sys, time\n"
            "os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'\n"
            "from huggingface_hub import snapshot_download\n"
            "for n in (1, 2, 3):\n"
            "    try:\n"
            "        snapshot_download(repo_id=%r, local_dir=%r, allow_patterns=%r, max_workers=8)\n"
            "        sys.exit(0)\n"
            "    except Exception as exc:\n"
            "        print('download attempt', n, 'failed:', type(exc).__name__, exc, flush=True)\n"
            "        time.sleep(5)\n"
            "sys.exit(1)\n"
        ) % (MODEL_REPO, CKPT_DIR, ALLOW_PATTERNS)
        run_streamed([sys.executable, "-c", download_code])
        stop.set()

    stamp("Step 6/7  encoding the prompt with the official T5 (alone on the GPU, weights memory-mapped)")
    if os.path.exists(EMBEDS_PATH):
        os.remove(EMBEDS_PATH)
    code = run_streamed([sys.executable, LAUNCHER_PATH, "--stage", "encode", "--repo", REPO_DIR, "--ckpt", CKPT_DIR,
                         "--embeds", EMBEDS_PATH, "--prompt=" + PROMPT], env=env)
    if code != 0 or not os.path.exists(EMBEDS_PATH):
        raise SystemExit("Prompt encoding failed (exit code %s). The traceback is printed above." % code)

    stamp("Step 7/7  generating with the official generate.py (task vace-1.3B, %s, portrait)" % SIZE)
    if os.path.exists(OUTPUT):
        os.remove(OUTPUT)

    def run_rung(frames, dtype):
        stamp("attempt: %s, %d frames (%.1fs at %d fps), %s" % (SIZE, frames, frames / FPS, FPS, dtype))
        cmd = [
            sys.executable, LAUNCHER_PATH, "--stage", "generate", "--repo", REPO_DIR, "--ckpt", CKPT_DIR,
            "--image", image_path, "--out", OUTPUT, "--embeds", EMBEDS_PATH, "--prompt=" + PROMPT,
            "--frames", str(frames), "--steps", str(NUM_STEPS), "--guidance", str(GUIDANCE_SCALE),
            "--seed", str(SEED), "--dtype", dtype,
        ]
        return run_streamed(cmd, env=env)

    outcome = run_ladder(build_ladder(FRAMES), run_rung)
    if outcome is None or not os.path.exists(OUTPUT):
        raise SystemExit("No setting produced a video. Read the messages above the last attempt; "
                         "if the GPU shows memory in use, Runtime > Disconnect and delete runtime, then Run all.")
    frames, dtype = outcome
    stamp("done: %s, %d frames, %s, total %.1f min" % (SIZE, frames, dtype, (time.time() - started) / 60))
    try:
        from IPython.display import Video, display
        display(Video(OUTPUT, embed=True, width=320))
    except Exception:
        pass
    try:
        from google.colab import files
        files.download(OUTPUT)
    except ImportError:
        print("saved to", OUTPUT)


main()